# MAMMAL Fine-Tuning with K-Fold Cross-Validation

Fine-tune MAMMAL once per fold and report the average, using **the same fold file** as the MMELON notebooks
(`train_cluster_cv_folds.npz` from [cluster_cv_folds.ipynb](cluster_cv_folds.ipynb)), so the two models are
compared on identical splits.

All folds are trained **side by side**, one epoch at a time: after each epoch every fold scores its own
held-out rows, and the metrics are averaged over folds. The epoch with the best average is selected, and
training stops early when `PATIENCE` epochs bring no improvement. Each fold keeps only its own best model.

**Environment:** run with the `biomed-mammal` kernel.

**Inputs:** the CSV written by `cluster_cv_folds.ipynb` and its `.npz` of fold indices.

**What to do with the result:** the number of epochs in `selection.json` is what
[MAMMAL_finetuning.ipynb](MAMMAL_finetuning.ipynb) should train the final model for. To score a test set with
all the fold models together, use [MAMMAL_inference_cv.ipynb](MAMMAL_inference_cv.ipynb).

**Output** (in `aircheck/output/`, which is gitignored):
```
output/
└── <DATASET>/<RUN_NAME>/
    ├── config.json
    ├── metrics_per_epoch.csv            # every fold and epoch
    ├── cv_summary.csv                   # mean and standard deviation over folds, per epoch
    ├── selection.json                   # the selected epoch and its averaged metrics
    └── fold_<i>/best_model/             # one fine-tuned model per fold (~1.8 GB each)
```

## 1. Inputs and settings

In [ ]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/
DATASET = "sample-data"

# Name of this run; outputs go to output/<DATASET>/<RUN_NAME>/ (must not already exist)
RUN_NAME = "mammal-cv"

# ── Data: the CSV and fold indices written by cluster_cv_folds.ipynb ──────────
DATA_DIR = Path("../output") / DATASET / "cv_folds"
DATA_PATH = DATA_DIR / "train_cluster_cv_data.csv"
CV_INDICES_PATH = DATA_DIR / "train_cluster_cv_folds.npz"   # the number of folds is read from it

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm/biomed.omics.bl.sm.ma-ted-458m"  # Hugging Face model ID

# ── LoRA ──────────────────────────────────────────────────────────────────────
# False: train every weight (needs ~15 GB of GPU memory).
# True:  freeze the model and train small added matrices instead (~4 GB, about 0.2% of the weights).
# Either way the saved model is a normal MAMMAL model: with LoRA the matrices are merged back in at
# the end, so inference is identical.
USE_LORA = False
LORA_R = 8                      # size of the added matrices; larger = more capacity, more memory
LORA_ALPHA = 16                 # how strongly they count; usually 2 x LORA_R
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q", "v"]  # attention query and value layers, the usual choice for T5

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 50                # maximum number of epochs
PATIENCE = 5               # stop after this many epochs without improvement
BATCH_SIZE = 16
LEARNING_RATE = 1e-5       # with USE_LORA, 1e-4 to 3e-4 usually works better
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.1      # share of all training steps spent warming the learning rate up
SEED = 42

# Balance the loss between the classes: label 0 gets weight 1, label 1 gets (number of 0s) / (number of 1s),
# counted on the training data only (in CV: on each fold's own training rows). False = unweighted loss.
BALANCE_CLASSES = True

# Prompt lengths, in tokens. A SMILES longer than MAX_SMILES_TOKENS is cut.
MAX_SMILES_TOKENS = 300
MAX_PROMPT_TOKENS = 320

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "pr_auc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

MAX_PER_CLASS = None      # e.g. 100 for a quick test run; None uses all data
NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes

In [ ]:
import json
import shutil

import numpy as np
import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "mammal").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-alignment repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
RUN_DIR = OUTPUT_DIR / DATASET / RUN_NAME
TOKENIZER_SOURCE = BASE_MODEL

for path in (DATA_PATH, CV_INDICES_PATH):
    assert Path(path).exists(), f"File not found: {path} (run cluster_cv_folds.ipynb first)"
assert not RUN_DIR.exists(), f"{RUN_DIR} already exists. Choose a new RUN_NAME or delete that folder."

cv_indices = np.load(CV_INDICES_PATH)
N_FOLDS = len([k for k in cv_indices.files if k.endswith("_train")])

# Disk: each fold keeps its training state between epochs (weights + optimizer, ~5.5 GB) and, once an
# epoch improves the average, a copy of it. At the end each copy becomes a ~1.8 GB model folder.
# With LoRA the best epoch is held in memory instead (a few MB), so only the training states are on disk.
state_gb = 5.5 * N_FOLDS
needed_gb = state_gb if USE_LORA else state_gb * 2
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, (
    f"Need ~{needed_gb:.0f} GB free for {N_FOLDS} folds, only {free_gb:.0f} GB available. "
    f"Use fewer folds, set USE_LORA = True, or free up space.")

RUN_DIR.mkdir(parents=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = dict(
    data_path=str(DATA_PATH), cv_indices_path=str(CV_INDICES_PATH), n_folds=N_FOLDS,
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL,
    use_lora=USE_LORA, lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODULES,
    epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, warmup_fraction=WARMUP_FRACTION, seed=SEED,
    hit_ks=HIT_KS, select_metrics=SELECT_METRICS, balance_classes=BALANCE_CLASSES,
    max_smiles_tokens=MAX_SMILES_TOKENS, max_prompt_tokens=MAX_PROMPT_TOKENS,
    max_per_class=MAX_PER_CLASS,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Run folder : {RUN_DIR}")
print(f"Folds      : {N_FOLDS}")
print(f"Training   : {'LoRA' if USE_LORA else 'all weights'}")
print(f"Disk       : ~{needed_gb:.0f} GB needed while training, {free_gb:.0f} GB free")
print(f"Device     : {DEVICE}")

## 2. Prepare the folds

Rows without a usable SMILES or label are dropped from each fold.

In [ ]:
import numpy as np
import pandas as pd
import torch
from functools import partial
from torch.utils.data import DataLoader, Dataset

import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.loggers import CSVLogger
from sklearn.metrics import average_precision_score, roc_auc_score
from peft import get_peft_model_state_dict, set_peft_model_state_dict

from fuse.data.utils.collates import CollateDefault
from fuse.dl.lightning.pl_module import LightningModuleDefault
from mammal.keys import *
from mammal.lora import get_lora_model
from mammal.losses import LossHead
from mammal.lr_schedulers import cosine_annealing_with_warmup_lr_scheduler
import pyarrow.parquet as pq
def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)
df = read_table(DATA_PATH, [SMILES_COL, LABEL_COL]).rename(
    columns={SMILES_COL: "smiles", LABEL_COL: "label"})
usable = df["smiles"].notna() & (df["smiles"].astype(str).str.strip() != "") & df["label"].notna()
assert set(df.loc[usable, "label"].astype(int).unique()) <= {0, 1}, "labels must be 0/1"


def limit_per_class(frame, n: int):
    parts = [g.sample(n=min(n, len(g)), random_state=SEED) for _, g in frame.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=SEED)


def positive_weight(labels) -> float:
    if not BALANCE_CLASSES:
        return 1.0
    n_neg, n_pos = int((labels == 0).sum()), int((labels == 1).sum())
    assert n_neg > 0 and n_pos > 0, "each fold needs both labels to balance the classes"
    return round(n_neg / n_pos, 4)


folds, summary = [], []
for i in range(N_FOLDS):
    train_idx = [j for j in cv_indices[f"fold{i}_train"] if usable.iloc[j]]
    test_idx = [j for j in cv_indices[f"fold{i}_test"] if usable.iloc[j]]
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()
    train_df["label"] = train_df["label"].astype(int)
    test_df["label"] = test_df["label"].astype(int)
    if MAX_PER_CLASS is not None:
        train_df, test_df = limit_per_class(train_df, MAX_PER_CLASS), limit_per_class(test_df, MAX_PER_CLASS)
    weight = positive_weight(train_df["label"])
    folds.append(dict(fold=i, train=train_df.reset_index(drop=True), test=test_df.reset_index(drop=True),
                      pos_weight=weight))
    summary.append(dict(fold=i, train=len(train_df), train_positives=int(train_df["label"].sum()),
                        test=len(test_df), test_positives=int(test_df["label"].sum()), pos_weight=weight))

pd.DataFrame(summary)

## 3. Load the tokenizer

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.model import Mammal

tokenizer_op = ModularTokenizerOp.from_pretrained(TOKENIZER_SOURCE)

NEG_TOKEN_ID = tokenizer_op.get_token_id("<0>")
POS_TOKEN_ID = tokenizer_op.get_token_id("<1>")
PAD_TOKEN_ID = tokenizer_op.get_token_id("<PAD>")
print(f"Tokenizer loaded. Answer tokens: <0> = {NEG_TOKEN_ID}, <1> = {POS_TOKEN_ID}")

## 4. Prompts, model and metrics

The same code as [MAMMAL_finetuning.ipynb](MAMMAL_finetuning.ipynb), applied once per fold.

In [ ]:
IGNORE_INDEX = -100
TOKEN_WEIGHTS_KEY = "data.token_weights"
MAX_ANSWER_TOKENS = 4
ANSWER_POSITION = 1  # the decoder writes <SENTINEL_ID_0> first, then the answer


def encoder_prompt(smiles: str) -> str:
    """The text MAMMAL reads for one molecule."""
    return (
        f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0>"
        f"<MOLECULAR_ENTITY><MOLECULAR_ENTITY_SMALL_MOLECULE>"
        f"<@TOKENIZER-TYPE=SMILES@MAX-LEN={MAX_SMILES_TOKENS}>"
        f"<SEQUENCE_NATURAL_START>{smiles}<SEQUENCE_NATURAL_END><EOS>"
    )


def prepare_sample(sample_dict: dict, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True) -> dict:
    """Tokenize one sample: the prompt, and (for training) the answer and its loss weight."""
    sample_dict[ENCODER_INPUTS_STR] = encoder_prompt(sample_dict["data.smiles"])
    tokenizer_op(
        sample_dict=sample_dict,
        key_in=ENCODER_INPUTS_STR,
        key_out_tokens_ids=ENCODER_INPUTS_TOKENS,
        key_out_attention_mask=ENCODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_PROMPT_TOKENS,
    )
    for k in (ENCODER_INPUTS_TOKENS, ENCODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    if not with_label:
        return sample_dict

    label = int(sample_dict["data.label"])

    # What the model should write
    sample_dict[LABELS_STR] = f"<@TOKENIZER-TYPE=SMILES><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=LABELS_STR,
        key_out_tokens_ids=LABELS_TOKENS, key_out_attention_mask=LABELS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (LABELS_TOKENS, LABELS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])
    # Padding must not count towards the loss
    sample_dict[LABELS_TOKENS][sample_dict[LABELS_TOKENS] == PAD_TOKEN_ID] = IGNORE_INDEX

    # What the decoder is fed during training (teacher forcing)
    sample_dict[DECODER_INPUTS_STR] = f"<@TOKENIZER-TYPE=SMILES><DECODER_START><SENTINEL_ID_0><{label}><EOS>"
    tokenizer_op(
        sample_dict=sample_dict, key_in=DECODER_INPUTS_STR,
        key_out_tokens_ids=DECODER_INPUTS_TOKENS, key_out_attention_mask=DECODER_INPUTS_ATTENTION_MASK,
        max_seq_len=MAX_ANSWER_TOKENS,
    )
    for k in (DECODER_INPUTS_TOKENS, DECODER_INPUTS_ATTENTION_MASK):
        sample_dict[k] = torch.tensor(sample_dict[k])

    # Class balancing: weight every counted position of this sample, 0 where the loss is ignored
    weight = pos_weight if label == 1 else 1.0
    sample_dict[TOKEN_WEIGHTS_KEY] = weight * (sample_dict[LABELS_TOKENS] != IGNORE_INDEX).float()
    return sample_dict


class MoleculeDataset(Dataset):
    """One row per molecule, from a DataFrame with 'smiles' and (for training) 'label' columns."""

    def __init__(self, df, tokenizer_op, pos_weight: float = 1.0, with_label: bool = True):
        self.smiles = df["smiles"].tolist()
        self.labels = df["label"].tolist() if with_label else [0] * len(df)
        self.tokenizer_op, self.pos_weight, self.with_label = tokenizer_op, pos_weight, with_label

    def __len__(self) -> int:
        return len(self.smiles)

    def __getitem__(self, idx: int) -> dict:
        sample = {"data.sample_id": idx, "data.smiles": self.smiles[idx], "data.label": self.labels[idx]}
        return prepare_sample(sample, self.tokenizer_op, self.pos_weight, self.with_label)


def make_collate():
    """Trim the padding each batch does not need (10-20% faster)."""
    return CollateDefault(special_handlers_keys={
        ENCODER_INPUTS_TOKENS: partial(CollateDefault.crop_padding, pad_token_id=PAD_TOKEN_ID),
        ENCODER_INPUTS_ATTENTION_MASK: partial(CollateDefault.crop_padding, pad_token_id=False),
    })
def compute_metrics(labels, scores, hit_ks=None, threshold=0.5) -> dict:
    """hit@K, AUROC, PR AUC and the threshold metrics, as the MMELON notebooks report them."""
    hit_ks = HIT_KS if hit_ks is None else hit_ks
    labels, scores = np.asarray(labels).astype(int), np.asarray(scores, dtype=float)
    top = np.argsort(-scores, kind="stable")
    metrics = {f"hit@{k}": int(labels[top[:k]].sum()) for k in hit_ks}
    pred = scores > threshold
    tp, fp = int((pred & (labels == 1)).sum()), int((pred & (labels == 0)).sum())
    tn, fn = int((~pred & (labels == 0)).sum()), int((~pred & (labels == 1)).sum())
    both_classes = len(set(labels.tolist())) == 2
    metrics.update(
        rocauc=float(roc_auc_score(labels, scores)) if both_classes else float("nan"),
        pr_auc=float(average_precision_score(labels, scores)) if both_classes else float("nan"),
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        specificity=tn / (tn + fp) if tn + fp else 0.0,
        accuracy=(tp + tn) / len(labels) if len(labels) else float("nan"),
    )
    return metrics
def build_model():
    """The pretrained model, wrapped in LoRA when USE_LORA is on."""
    model = Mammal.from_pretrained(BASE_MODEL)
    if USE_LORA:
        model = get_lora_model(model, r=LORA_R, lora_alpha=LORA_ALPHA,
                               lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
    else:
        print(f"Full fine-tuning: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M trainable parameters")
    return model


def save_model(model, path: Path, lora_state: dict | None = None):
    """Save as a plain MAMMAL model, so inference is the same with and without LoRA.

    With LoRA the added matrices are merged into the weights first; `lora_state` restores the best
    epoch's matrices before merging (the model in memory holds the last epoch's)."""
    path.mkdir(parents=True, exist_ok=True)
    if USE_LORA:
        if lora_state is not None:
            set_peft_model_state_dict(model, lora_state)
        merged = model.merge_and_unload()    # gives back a plain Mammal
        merged._save_pretrained(str(path))
    else:
        model._save_pretrained(str(path))
    tokenizer_op.save_pretrained(str(path / "tokenizer"))
class HitAtKModule(LightningModuleDefault):
    """MAMMAL's training module, plus the metrics the MMELON notebooks report.

    The score for a molecule is the probability the model gives to `<1>` at the answer position,
    against `<0>`: the same number the inference notebook computes."""

    @property
    def model(self):
        """The MAMMAL model inside the module (fuse stores it as `_model`)."""
        return self._model

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch_dict, batch_idx, dataloader_idx=0):
        super().validation_step(batch_dict, batch_idx, dataloader_idx)
        probs = batch_dict[SCORES][:, ANSWER_POSITION, :]          # (batch, vocabulary)
        pos, neg = probs[:, POS_TOKEN_ID], probs[:, NEG_TOKEN_ID]
        self._val_scores.append((pos / (pos + neg + 1e-8)).detach().float().cpu())
        answers = batch_dict[LABELS_TOKENS][:, ANSWER_POSITION]
        self._val_labels.append((answers == POS_TOKEN_ID).detach().int().cpu())

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores = torch.cat(self._val_scores).numpy()
        labels = torch.cat(self._val_labels).numpy()
        for name, value in compute_metrics(labels, scores).items():
            self.log(f"val_{name}", float(value))
        # The loss Lightning logged under its own name, copied to the val_ name the selection uses
        logged = self.trainer.callback_metrics.get("validation.losses.total_loss")
        self.log("val_loss", float(logged) if logged is not None else float("nan"))


def make_optimizers(total_steps: int):
    """AdamW with a warmup then a cosine decay."""
    warmup_steps = max(1, int(WARMUP_FRACTION * total_steps))

    def configure(module):
        opt = torch.optim.AdamW([p for p in module.trainer.model.parameters() if p.requires_grad],
                                lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        sched = cosine_annealing_with_warmup_lr_scheduler(
            opt, T_max=max(1, total_steps - warmup_steps), num_warmup_steps=warmup_steps, eta_min_factor=0.1)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "step"}}

    return configure, warmup_steps


LOSSES = lambda: {"ce": LossHead(loss_type="ce", sample_token_weights_key=TOKEN_WEIGHTS_KEY)}

pl.seed_everything(SEED, workers=True)

for f in folds:
    f["dir"] = RUN_DIR / f"fold_{f['fold']}"
    f["dir"].mkdir(parents=True, exist_ok=True)
    f["loaders"] = (
        DataLoader(MoleculeDataset(f["train"], tokenizer_op, pos_weight=f["pos_weight"]),
                   batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate(),
                   num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0),
        DataLoader(MoleculeDataset(f["test"], tokenizer_op, pos_weight=f["pos_weight"]),
                   batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(),
                   num_workers=NUM_WORKERS, persistent_workers=NUM_WORKERS > 0),
    )


def make_module(f):
    """A fresh model and training module for one fold (its weights are restored from disk by the trainer)."""
    configure_optimizers, _ = make_optimizers(len(f["loaders"][0]) * EPOCHS)
    return HitAtKModule(model=build_model(), losses=LOSSES(),
                        optimizers_and_lr_schs=configure_optimizers, model_dir=str(f["dir"]))


print(f"{N_FOLDS} folds ready: " + ", ".join(f"fold {f['fold']} {len(f['loaders'][0])} batches"
                                             for f in folds))

## 5. Train the folds side by side

Every fold runs one epoch, then all folds are scored and the metrics averaged. The averaged metrics decide
when to stop, so the folds always stop together on the same epoch.

In [ ]:
%%time
import gc
import time

allowed = [f"hit@{k}" for k in HIT_KS] + ["rocauc", "pr_auc", "precision", "recall", "specificity",
                                          "accuracy", "loss"]
assert set(SELECT_METRICS) <= set(allowed), f"SELECT_METRICS must come from {allowed}"

LOG = RUN_DIR / "progress.log"   # follow a running job with: tail -f <run folder>/progress.log


def log(message: str):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {message}"
    print(line, flush=True)
    with open(LOG, "a") as handle:
        handle.write(line + "\n")


def save_metrics(rows):
    # Write the per-epoch metrics so far, so they survive an interrupted run
    df = pd.DataFrame(rows)
    hits = [f"hit@{k}" for k in HIT_KS]
    names = [c for c in hits if c in df.columns] + [c for c in df.columns
                                                    if c not in hits and c not in ("fold", "epoch")]
    df = df[["fold", "epoch"] + names]
    df.to_csv(RUN_DIR / "metrics_per_epoch.csv", index=False)
    for f in folds:
        df[df["fold"] == f["fold"]].to_csv(f["dir"] / "metrics_per_epoch.csv", index=False)
    return df, names


rows, best_key, BEST_EPOCH, wait = [], None, None, 0
log(f"Training {N_FOLDS} folds side by side, up to {EPOCHS} epochs (patience {PATIENCE})")

for epoch in range(EPOCHS):
    for f in folds:
        state_path = f["dir"] / "training_state.ckpt"
        pl.seed_everything(SEED + 1000 * f["fold"] + epoch, verbose=False)
        start = time.time()
        trainer = pl.Trainer(
            max_epochs=epoch + 1, logger=False, enable_checkpointing=False, default_root_dir=f["dir"],
            accelerator="auto", devices=1, num_sanity_val_steps=0,
            enable_model_summary=False, enable_progress_bar=False,
        )
        module = make_module(f)
        # Continue this fold from its previous epoch (weights and optimizer state are restored)
        trainer.fit(module, *f["loaders"], ckpt_path=state_path if epoch > 0 else None)
        trainer.save_checkpoint(state_path)
        rows.append(dict(fold=f["fold"], epoch=epoch, **{
            k.removeprefix("val_"): float(v)
            for k, v in trainer.callback_metrics.items() if k.startswith("val_")
        }))
        if USE_LORA:
            f["last_lora"] = {k: v.detach().cpu().clone()
                              for k, v in get_peft_model_state_dict(module.model).items()}
        log(f"epoch {epoch}, fold {f['fold'] + 1}/{N_FOLDS} done in {(time.time() - start) / 60:.1f} min")
        del trainer, module
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    this_epoch = pd.DataFrame([r for r in rows if r["epoch"] == epoch])
    means = this_epoch.mean(numeric_only=True)
    # Larger is better; loss is negated so a lower loss ranks higher. Ties keep the earlier epoch.
    key = tuple(float(means[m]) * (-1 if m == "loss" else 1) for m in SELECT_METRICS)
    if best_key is None or key > best_key:
        best_key, BEST_EPOCH, wait = key, epoch, 0
        for f in folds:
            if USE_LORA:
                f["best_lora"] = f["last_lora"]          # a few MB, kept in memory
            else:
                shutil.copy2(f["dir"] / "training_state.ckpt", f["dir"] / "best_state.ckpt")
        status = "best so far, state kept"
    else:
        wait += 1
        status = f"no improvement ({wait}/{PATIENCE})"

    log(f"epoch {epoch:3d} mean over folds:  "
        + "  ".join(f"{m} {means[m]:.4f}" for m in SELECT_METRICS) + f"  -> {status}")
    save_metrics(rows)
    if wait >= PATIENCE:
        log(f"Stopping early: no improvement of the fold average for {PATIENCE} epochs")
        break

EPOCHS_RUN = epoch + 1
cv_metrics, metric_names = save_metrics(rows)
log(f"Trained {EPOCHS_RUN} epochs; selected epoch {BEST_EPOCH}")

## 6. Save the fold models

Each fold's best epoch becomes a plain MAMMAL model folder, the same format a single fine-tuning run
produces, so [MAMMAL_inference_cv.ipynb](MAMMAL_inference_cv.ipynb) can load them.

In [ ]:
# Turn each fold's best epoch into a plain MAMMAL model folder, then drop the training states
for f in folds:
    model = build_model()
    if not USE_LORA:
        state = torch.load(f["dir"] / "best_state.ckpt", map_location="cpu")["state_dict"]
        # Lightning stores the module's weights under "_model."; strip that to get the model's own keys
        state = {k.split(".", 1)[1] if k.startswith("_model.") else k: v for k, v in state.items()}
        result = model.load_state_dict(state, strict=False)
        assert not result.missing_keys, f"fold {f['fold']}: missing weights {result.missing_keys[:5]}"
    save_model(model, f["dir"] / "best_model",
               lora_state=f["best_lora"] if USE_LORA else None)
    del model
    gc.collect()

for f in folds:
    for name in ("training_state.ckpt", "best_state.ckpt"):
        path = f["dir"] / name
        if path.exists():
            path.unlink()

print(f"Saved {N_FOLDS} fold models (epoch {BEST_EPOCH}) and removed the training states")

## 7. Cross-validation summary and the selected epoch

In [ ]:
cv_summary = cv_metrics.groupby("epoch")[metric_names].agg(["mean", "std"])
cv_summary.to_csv(RUN_DIR / "cv_summary.csv")

best = cv_metrics[cv_metrics["epoch"] == BEST_EPOCH]
selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=BEST_EPOCH,
    epochs_trained=EPOCHS_RUN,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    stopped_early=EPOCHS_RUN < EPOCHS,
    n_folds=N_FOLDS,
    use_lora=USE_LORA,
    cv_metrics_mean={m: float(best[m].mean()) for m in metric_names},
    cv_metrics_std={m: float(best[m].std()) for m in metric_names},
    models=[str(f["dir"] / "best_model") for f in folds],
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))

print(f"Selected epoch {BEST_EPOCH} of {EPOCHS_RUN} trained")
print(f"Train the final model for {BEST_EPOCH + 1} epochs in MAMMAL_finetuning.ipynb\n")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best[m].mean():.4f} +/- {best[m].std():.4f}")
display(cv_summary.round(4))

In [ ]:
# Files produced by this run
total = 0
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        size = path.stat().st_size / 1e6
        total += size
        if size > 1 or path.suffix in (".csv", ".json"):
            print(f"{str(path.relative_to(RUN_DIR)):60s} {size:8.2f} MB")
print(f"\n{'total':60s} {total:8.2f} MB")